## 2.1.  CBOW ve Skip-Gram 

### 2.1.1 Kurulum Hazırlığı

In [1]:
import numpy as np

# --- ORTAK KURULUM ---
print("--- GENEL KURULUM BAŞLIYOR ---")
# Basit bir metin korpusu oluşturalım.
# İlişkili kelimeler (kral-adam, kraliçe-kadın) var.
corpus_text = "kral, güçlü bir adamdır. kraliçe, bilge bir kadındır. adam, kral ile yaşar. kadın, kraliçe ile yaşar."
corpus = corpus_text.lower().replace('.', '').replace(',', '').split()

print(f"Metin Korpusu: {corpus}\n")

# Kelime dağarcığı (vocabulary) oluşturma
vocab = sorted(list(set(corpus)))
word_to_ix = {word: i for i, word in enumerate(vocab)}
ix_to_word = {i: word for i, word in enumerate(vocab)}
vocab_size = len(vocab)

print(f"Kelime Dağarcığı (Vocabulary): {vocab}")
print(f"Kelime -> Index Sözlüğü: {word_to_ix}")
print(f"Vocabulary Boyutu: {vocab_size}\n")

# Model hiperparametreleri
embedding_dim = 10  # Her kelimeyi temsil edecek vektörün boyutu
window_size = 2   # Bağlam penceresinin boyutu (hedef kelimenin sağı ve solundan kaç kelime alınacağı)

# Simülasyon için rastgele Embedding matrisleri (Gerçekte bunlar eğitimle öğrenilir)
# W1: Girdi -> Gizli Katman (Embedding'lerin tutulduğu yer)
# W2: Gizli Katman -> Çıktı
np.random.seed(42)
W1 = np.random.rand(vocab_size, embedding_dim)
W2 = np.random.rand(embedding_dim, vocab_size)

print("Simülasyon için rastgele W1 (Embedding) ve W2 (Çıktı) matrisleri oluşturuldu.")
print("="*60)

print('W1', W1)
print('W2', W2)

--- GENEL KURULUM BAŞLIYOR ---
Metin Korpusu: ['kral', 'güçlü', 'bir', 'adamdır', 'kraliçe', 'bilge', 'bir', 'kadındır', 'adam', 'kral', 'ile', 'yaşar', 'kadın', 'kraliçe', 'ile', 'yaşar']

Kelime Dağarcığı (Vocabulary): ['adam', 'adamdır', 'bilge', 'bir', 'güçlü', 'ile', 'kadın', 'kadındır', 'kral', 'kraliçe', 'yaşar']
Kelime -> Index Sözlüğü: {'adam': 0, 'adamdır': 1, 'bilge': 2, 'bir': 3, 'güçlü': 4, 'ile': 5, 'kadın': 6, 'kadındır': 7, 'kral': 8, 'kraliçe': 9, 'yaşar': 10}
Vocabulary Boyutu: 11

Simülasyon için rastgele W1 (Embedding) ve W2 (Çıktı) matrisleri oluşturuldu.
W1 [[0.37454012 0.95071431 0.73199394 0.59865848 0.15601864 0.15599452
  0.05808361 0.86617615 0.60111501 0.70807258]
 [0.02058449 0.96990985 0.83244264 0.21233911 0.18182497 0.18340451
  0.30424224 0.52475643 0.43194502 0.29122914]
 [0.61185289 0.13949386 0.29214465 0.36636184 0.45606998 0.78517596
  0.19967378 0.51423444 0.59241457 0.04645041]
 [0.60754485 0.17052412 0.06505159 0.94888554 0.96563203 0.80839735
 

### 2.1.2 CBoW

In [2]:
# --- BÖLÜM 1: CBOW MODELİ SİMÜLASYONU ---
print("\n--- BÖLÜM 1: CBOW (Continuous Bag-of-Words) ---")
print(f"Amaç: ['çevre_kelime_1', 'çevre_kelime_2'] -> 'hedef_kelime' tahmin etmek.\n")

# Adım 1.1: CBOW için eğitim verisi oluşturma
# (Bağlam, Hedef) çiftleri oluşturulur.
cbow_data = []
for i in range(window_size, len(corpus) - window_size):
    context_indices = []
    # Sol pencere
    for j in range(window_size, 0, -1):
        context_indices.append(word_to_ix[corpus[i-j]])
    # Sağ pencere
    for j in range(1, window_size + 1):
        context_indices.append(word_to_ix[corpus[i+j]])
    
    target_index = word_to_ix[corpus[i]]
    cbow_data.append((context_indices, target_index))

print("Adım 1.1: CBOW için Eğitim Verisi Oluşturuldu (context_indices, target_index):")
print("Örnek Veriler:")
for context, target in cbow_data[:3]:
    context_words = [ix_to_word[ix] for ix in context]
    target_word = ix_to_word[target]
    print(f"   Bağlam: {context_words}  --->  Hedef: '{target_word}'")
print("...\n")


# Adım 1.2: Bir CBOW örneği için ileri yayılım (Forward Pass) simülasyonu
print("Adım 1.2: Tek bir örnek üzerinden CBOW'un çalışma mantığı:\n")
# Örnek alalım: (['kral', 'güçlü', 'bir', 'adamdır'], 'bir') -> hedef 'bir' olmasın, hedef 'adamdır', bağlam 'güçlü bir' olsun.
# Pencereyi 1 yapalım daha anlaşılır olsun
window_size_cbow = 1
# Yeni veri:
# (['güçlü', 'adamdır'], 'bir')
context_sample_ix = [word_to_ix['güçlü'], word_to_ix['adamdır']]
target_sample_ix = word_to_ix['bir']
context_sample_words = [ix_to_word[ix] for ix in context_sample_ix]
target_sample_word = ix_to_word[target_sample_ix]

print(f"Seçilen Örnek: Bağlam: {context_sample_words} -> Hedef: '{target_sample_word}'\n")

# 1. Bağlam kelimelerinin embedding vektörlerini W1 matrisinden al.
print("1. Bağlam kelimelerinin embedding vektörleri W1'den alınır.")
context_vectors = W1[context_sample_ix]
# print(f"   '{context_sample_words[0]}' vektörü: {np.round(context_vectors[0], 2)}")
# print(f"   '{context_sample_words[1]}' vektörü: {np.round(context_vectors[1], 2)}\n")

# 2. Bu vektörlerin ortalamasını alarak tek bir bağlam vektörü oluştur.
print("2. Bu vektörlerin ortalaması alınarak tek bir 'bağlam vektörü' (h) oluşturulur.")
context_vector_h = np.mean(context_vectors, axis=0)
# print(f"   Ortalama Bağlam Vektörü (h): {np.round(context_vector_h, 2)}\n")

# 3. Bu bağlam vektörünü W2 matrisi ile çarparak skorları elde et.
print("3. Bu ortalama vektör, W2 matrisi ile çarpılarak tüm kelimeler için skorlar üretilir.")
scores = np.dot(context_vector_h, W2)

# 4. Softmax ile skorları olasılıklara çevir.
print("\n4. Softmax fonksiyonu ile bu skorlar bir olasılık dağılımına dönüştürülür.")
probabilities = np.exp(scores) / np.sum(np.exp(scores))

# 5. Sonuç
print("\n5. Modelin amacı, eğitim sırasında W1 ve W2'yi güncelleyerek,")
print(f"doğru hedef kelime olan '{target_sample_word}' için bu olasılığı maksimize etmektir.")
print(f"   Modelin '{target_sample_word}' için ürettiği olasılık (eğitim öncesi): {probabilities[target_sample_ix]:.4f}")
print("="*60)


--- BÖLÜM 1: CBOW (Continuous Bag-of-Words) ---
Amaç: ['çevre_kelime_1', 'çevre_kelime_2'] -> 'hedef_kelime' tahmin etmek.

Adım 1.1: CBOW için Eğitim Verisi Oluşturuldu (context_indices, target_index):
Örnek Veriler:
   Bağlam: ['kral', 'güçlü', 'adamdır', 'kraliçe']  --->  Hedef: 'bir'
   Bağlam: ['güçlü', 'bir', 'kraliçe', 'bilge']  --->  Hedef: 'adamdır'
   Bağlam: ['bir', 'adamdır', 'bilge', 'bir']  --->  Hedef: 'kraliçe'
...

Adım 1.2: Tek bir örnek üzerinden CBOW'un çalışma mantığı:

Seçilen Örnek: Bağlam: ['güçlü', 'adamdır'] -> Hedef: 'bir'

1. Bağlam kelimelerinin embedding vektörleri W1'den alınır.
2. Bu vektörlerin ortalaması alınarak tek bir 'bağlam vektörü' (h) oluşturulur.
3. Bu ortalama vektör, W2 matrisi ile çarpılarak tüm kelimeler için skorlar üretilir.

4. Softmax fonksiyonu ile bu skorlar bir olasılık dağılımına dönüştürülür.

5. Modelin amacı, eğitim sırasında W1 ve W2'yi güncelleyerek,
doğru hedef kelime olan 'bir' için bu olasılığı maksimize etmektir.
   Modeli

### 2.1.3 Skip-Gram ile komşu tahmini

In [3]:
# --- BÖLÜM 2: SKIP-GRAM MODELİ SİMÜLASYONU ---
print("\n--- BÖLÜM 2: Skip-gram ---")
print(f"Amaç: 'hedef_kelime' -> ['çevre_kelime_1', 'çevre_kelime_2'] tahmin etmek.\n")

# Adım 2.1: Skip-gram için eğitim verisi oluşturma
# (Hedef, Bağlam_Kelimesi) çiftleri oluşturulur.
# Bir hedef kelime, birden çok eğitim örneği üretir.
skipgram_data = []
for i in range(window_size, len(corpus) - window_size):
    center_word_ix = word_to_ix[corpus[i]]
    # Sol ve sağ penceredeki her kelime için ayrı bir çift oluştur
    for j in range(window_size, 0, -1):
        context_word_ix = word_to_ix[corpus[i-j]]
        skipgram_data.append((center_word_ix, context_word_ix))
    for j in range(1, window_size + 1):
        context_word_ix = word_to_ix[corpus[i+j]]
        skipgram_data.append((center_word_ix, context_word_ix))

print("Adım 2.1: Skip-gram için Eğitim Verisi Oluşturuldu (hedef_index, context_index):")
print("Örnek Veriler:")
for center, context in skipgram_data[:5]:
    center_word = ix_to_word[center]
    context_word = ix_to_word[context]
    print(f"   Hedef: '{center_word}'  --->  Tahmin Edilecek Bağlam: '{context_word}'")
print("...\n")

# Adım 2.2: Bir Skip-gram örneği için ileri yayılım (Forward Pass) simülasyonu
print("Adım 2.2: Tek bir örnek üzerinden Skip-gram'in çalışma mantığı:\n")
# Örnek alalım: ('güçlü', 'kral')
center_sample_ix = word_to_ix['güçlü']
context_sample_ix = word_to_ix['kral']
center_sample_word = ix_to_word[center_sample_ix]
context_sample_word = ix_to_word[context_sample_ix]

print(f"Seçilen Örnek: Hedef: '{center_sample_word}' -> Tahmin Edilecek Bağlam: '{context_sample_word}'\n")

# 1. Hedef kelimenin embedding vektörünü W1 matrisinden al.
print("1. Hedef kelimenin embedding vektörü W1'den alınır.")
center_vector = W1[center_sample_ix]
# print(f"   '{center_sample_word}' vektörü: {np.round(center_vector, 2)}\n")

# 2. Bu vektörü W2 matrisi ile çarparak skorları elde et. (CBOW'dan farkı: ortalama yok!)
print("2. Bu tek vektör, W2 matrisi ile çarpılarak tüm kelimeler için skorlar üretilir.")
scores = np.dot(center_vector, W2)

# 3. Softmax ile skorları olasılıklara çevir.
print("\n3. Softmax fonksiyonu ile bu skorlar bir olasılık dağılımına dönüştürülür.")
probabilities = np.exp(scores) / np.sum(np.exp(scores))

# 4. Sonuç
print("\n4. Modelin amacı, eğitim sırasında W1 ve W2'yi güncelleyerek,")
print(f"doğru bağlam kelimesi olan '{context_sample_word}' için bu olasılığı maksimize etmektir.")
print(f"Bu işlem, '{center_sample_word}' kelimesinin tüm bağlam kelimeleri için tekrarlanır.")
print(f"   Modelin '{context_sample_word}' için ürettiği olasılık (eğitim öncesi): {probabilities[context_sample_ix]:.4f}")
print("="*60)


--- BÖLÜM 2: Skip-gram ---
Amaç: 'hedef_kelime' -> ['çevre_kelime_1', 'çevre_kelime_2'] tahmin etmek.

Adım 2.1: Skip-gram için Eğitim Verisi Oluşturuldu (hedef_index, context_index):
Örnek Veriler:
   Hedef: 'bir'  --->  Tahmin Edilecek Bağlam: 'kral'
   Hedef: 'bir'  --->  Tahmin Edilecek Bağlam: 'güçlü'
   Hedef: 'bir'  --->  Tahmin Edilecek Bağlam: 'adamdır'
   Hedef: 'bir'  --->  Tahmin Edilecek Bağlam: 'kraliçe'
   Hedef: 'adamdır'  --->  Tahmin Edilecek Bağlam: 'güçlü'
...

Adım 2.2: Tek bir örnek üzerinden Skip-gram'in çalışma mantığı:

Seçilen Örnek: Hedef: 'güçlü' -> Tahmin Edilecek Bağlam: 'kral'

1. Hedef kelimenin embedding vektörü W1'den alınır.
2. Bu tek vektör, W2 matrisi ile çarpılarak tüm kelimeler için skorlar üretilir.

3. Softmax fonksiyonu ile bu skorlar bir olasılık dağılımına dönüştürülür.

4. Modelin amacı, eğitim sırasında W1 ve W2'yi güncelleyerek,
doğru bağlam kelimesi olan 'kral' için bu olasılığı maksimize etmektir.
Bu işlem, 'güçlü' kelimesinin tüm bağla

### 2.1.4 CBoW ve Skip-Gram Birlikteliği

In [10]:
import gensim
from gensim.models import Word2Vec
import warnings
warnings.filterwarnings("ignore") # Gereksiz uyarıları gizle

# --- Adım 1: Örnek Metin Korpusu Hazırlama ---
print("--- Adım 1: Veri Hazırlığı ---")
# Metin verimiz, cümlelerin token'lara (kelimelere) ayrılmış bir listesi olmalı.
corpus_text = "kral, güçlü bir adamdır. kraliçe, bilge bir kadındır. adam, kral ile yaşar. kadın, kraliçe ile yaşar."
# Cümlelere ve kelimelere ayıralım
corpus = [gensim.utils.simple_preprocess(sentence) for sentence in corpus_text.split('.')]

print("Token'lara ayrılmış metin (liste içinde liste yapısı):")
for sentence in corpus:
    if sentence: # Boş listeleri atla
        print(f"   {sentence}")
print("\n")


# --- Adım 2: CBOW (sg=0) ile Model Eğitimi ---
print("="*60)
print("--- BÖLÜM 2: CBOW Modeli (sg=0 - Varsayılan) ---")

# Modeli oluştururken sg=0 parametresini belirtiyoruz (belirtmesek de varsayılan bu olurdu).
# vector_size: embedding boyutu
# window: bağlam penceresi
# min_count: kelimenin dikkate alınması için minimum geçme sayısı
# workers: eğitim için kullanılacak işlemci çekirdeği sayısı (tekrarlanabilirlik için 1)
cbow_model = Word2Vec(sentences=corpus, vector_size=100, window=2, min_count=1, sg=0, workers=1)

print("\nCBOW modeli başarıyla eğitildi.")
print("Şimdi CBOW modelinin öğrendiği ilişkileri test edelim:\n")

# 'kral' kelimesinin vektör temsilini alalım
try:
    vec_kral_cbow = cbow_model.wv['kral']
    print(f"'kral' kelimesinin CBOW vektörünün ilk 5 boyutu: {vec_kral_cbow[:5]}")
except KeyError:
    print("'kral' kelimesi sözlükte bulunamadı.")

# Benzerlik testi
try:
    similarity_cbow = cbow_model.wv.similarity('kral', 'adam')
    print(f"CBOW - Benzerlik('kral', 'adam'): {similarity_cbow:.4f}")
except KeyError:
    print("Test kelimelerinden biri sözlükte bulunamadı.")

# En benzer kelimeleri bulma
try:
    most_similar_cbow = cbow_model.wv.most_similar('kraliçe')
    print(f"CBOW - 'kraliçe' kelimesine en benzer kelimeler: {most_similar_cbow}")
except KeyError:
    print("'kraliçe' kelimesi sözlükte bulunamadı.")


# --- Adım 3: Skip-gram (sg=1) ile Model Eğitimi ---
print("\n" + "="*60)
print("--- BÖLÜM 3: Skip-gram Modeli (sg=1) ---")

# Bu sefer, diğer tüm parametreler aynı kalacak şekilde sg=1 olarak değiştiriyoruz.
skipgram_model = Word2Vec(sentences=corpus, vector_size=100, window=2, min_count=1, sg=1, workers=1)

print("\nSkip-gram modeli başarıyla eğitildi.")
print("Şimdi Skip-gram modelinin öğrendiği ilişkileri test edelim:\n")

# 'kral' kelimesinin vektör temsilini alalım
try:
    vec_kral_skipgram = skipgram_model.wv['kral']
    print(f"'kral' kelimesinin Skip-gram vektörünün ilk 5 boyutu: {vec_kral_skipgram[:5]}")
except KeyError:
    print("'kral' kelimesi sözlükte bulunamadı.")

# Benzerlik testi
try:
    similarity_skipgram = skipgram_model.wv.similarity('kral', 'adam')
    print(f"Skip-gram - Benzerlik('kral', 'adam'): {similarity_skipgram:.4f}")
except KeyError:
    print("Test kelimelerinden biri sözlükte bulunamadı.")

# En benzer kelimeleri bulma
try:
    most_similar_skipgram = skipgram_model.wv.most_similar('kraliçe')
    print(f"Skip-gram - 'kraliçe' kelimesine en benzer kelimeler: {most_similar_skipgram}")
except KeyError:
    print("'kraliçe' kelimesi sözlükte bulunamadı.")


# --- Adım 4: Sonuç ve Yorum ---
print("\n" + "="*60)
print("--- BÖLÜM 4: Sonuç ve Yorum ---")
print("Gördüğünüz gibi, aynı veri üzerinde, sadece `sg` parametresini değiştirerek iki farklı model eğittik.")
print("\nİki modelin ürettiği sonuçlar arasındaki farklar şunlardan kaynaklanır:")
print("- Farklı Algoritmalar: CBOW, bağlamın ortalamasını alarak daha 'yumuşak' ve genel bir temsil öğrenir.")
print("- Detay Seviyesi: Skip-gram, her (hedef, bağlam) kelime çiftini ayrı bir görev olarak ele aldığı için daha fazla veri üzerinde çalışır ve potansiyel olarak daha ince detayları ve nadir kelimelerin ilişkilerini yakalayabilir.")
print("\nPratikte Hangisi Seçilir?")
print("- Hız gerekliyse ve büyük bir veri setiniz varsa: CBOW (`sg=0`) genellikle daha hızlıdır.")
print("- Vektör kalitesi çok önemliyse veya veri setiniz küçükse/nadir kelimeler içeriyorsa: Skip-gram (`sg=1`) genellikle daha iyi sonuçlar verir.")
print("\nGensim, bu seçimi tek bir parametre ile kolayca yapmanızı sağlayarak büyük bir esneklik sunar.")

--- Adım 1: Veri Hazırlığı ---
Token'lara ayrılmış metin (liste içinde liste yapısı):
   ['kral', 'güçlü', 'bir', 'adamdır']
   ['kraliçe', 'bilge', 'bir', 'kadındır']
   ['adam', 'kral', 'ile', 'yaşar']
   ['kadın', 'kraliçe', 'ile', 'yaşar']


--- BÖLÜM 2: CBOW Modeli (sg=0 - Varsayılan) ---

CBOW modeli başarıyla eğitildi.
Şimdi CBOW modelinin öğrendiği ilişkileri test edelim:

'kral' kelimesinin CBOW vektörünün ilk 5 boyutu: [-0.00713902  0.00124103 -0.00717672 -0.00224462  0.0037193 ]
CBOW - Benzerlik('kral', 'adam'): 0.0348
CBOW - 'kraliçe' kelimesine en benzer kelimeler: [('bilge', 0.1991206258535385), ('kral', 0.17018885910511017), ('adam', 0.145950585603714), ('kadın', 0.06408977508544922), ('kadındır', -0.002754019573330879), ('bir', -0.013514922931790352), ('ile', -0.023671654984354973), ('adamdır', -0.03284316137433052), ('yaşar', -0.05234673619270325), ('güçlü', -0.1019841730594635)]

--- BÖLÜM 3: Skip-gram Modeli (sg=1) ---

Skip-gram modeli başarıyla eğitildi.
Şimdi Skip

### 2.2. Word2Vec (Gensim) Basitleştirilmiş Örnek

In [11]:
from gensim.models import Word2Vec

# 1. Eğitim verisi: küçük ve sembolik bir metin kümesi
corpus = [
    ["king", "is", "a", "strong", "man"],
    ["queen", "is", "a", "wise", "woman"],
    ["man", "and", "woman", "are", "human"],
    ["king", "and", "queen", "rule", "the", "kingdom"],
    ["queen", "rules", "with", "wisdom"],
    ["man", "is", "mortal"],
    ["woman", "is", "mortal"],
    ["turkcell","ile","baglan","hayata"]
]

# 2. Word2Vec modelini eğit (skip-gram = sg=1)
model = Word2Vec(sentences=corpus, vector_size=50, window=2, min_count=1, sg=1)

# 3. Örnek: Benzer kelimeleri bulma
print("Benzer kelimeler (king):")
for word, score in model.wv.most_similar("king"):
    print(f"  {word:>8} → {score:.3f}")

# 4. Anlamsal çıkarım örneği: king - man + woman ≈ ?
print("\nAnlamsal Analojiler:")
for word, score in model.wv.most_similar(positive=["king", "woman"], negative=["man"]):
    print(f"  {word:>8} → {score:.3f}")

# 5. Örnek: Benzer kelimeleri bulma : turkcell
print("Benzer kelimeler (turkcell):")
for word, score in model.wv.most_similar("turkcell"):
    print(f"  {word:>8} → {score:.3f}")
'''
# 6. Örnek: Benzer kelimeleri bulma : turkcell
print("Benzer kelimeler (hascelik):")
for word, score in model.wv.most_similar("hascelik"):
    print(f"  {word:>8} → {score:.3f}")
'''

Benzer kelimeler (king):
    hayata → 0.185
      rule → 0.166
    mortal → 0.137
        is → 0.132
      with → 0.118
    wisdom → 0.115
         a → 0.113
      wise → 0.112
  turkcell → 0.096
       man → 0.045

Anlamsal Analojiler:
    mortal → 0.274
       are → 0.212
      with → 0.179
       and → 0.151
       the → 0.143
   kingdom → 0.138
         a → 0.135
      rule → 0.117
    strong → 0.114
        is → 0.111
Benzer kelimeler (turkcell):
     rules → 0.530
       the → 0.226
       and → 0.198
      with → 0.190
    wisdom → 0.174
        is → 0.167
     queen → 0.156
      wise → 0.111
      king → 0.096
      rule → 0.070


'\n# 6. Örnek: Benzer kelimeleri bulma : turkcell\nprint("Benzer kelimeler (hascelik):")\nfor word, score in model.wv.most_similar("hascelik"):\n    print(f"  {word:>8} → {score:.3f}")\n'

In [12]:
print('king', 'queen', model.wv.similarity('king', 'queen'))
print('wisdom', 'rules', model.wv.similarity('wisdom', 'rules'))

king queen -0.21871693
wisdom rules 0.18846765


In [13]:
import numpy as np
embedding_matrix = np.array([model.wv[word] for word in model.wv.index_to_key])

# Shape kontrolü
print(f"Embedding matris boyutu: {embedding_matrix.shape}")  # (vocab_size, vector_size)

# İlk 5 kelimenin embedding'ini yazdır
for i, word in enumerate(model.wv.index_to_key[:5]):
    print(f"{word}: {embedding_matrix[i]}")

Embedding matris boyutu: (22, 50)
is: [-1.0724545e-03  4.7286271e-04  1.0206699e-02  1.8018546e-02
 -1.8605899e-02 -1.4233618e-02  1.2917745e-02  1.7945977e-02
 -1.0030856e-02 -7.5267432e-03  1.4761009e-02 -3.0669428e-03
 -9.0732267e-03  1.3108104e-02 -9.7203208e-03 -3.6320353e-03
  5.7531595e-03  1.9837476e-03 -1.6570430e-02 -1.8897636e-02
  1.4623532e-02  1.0140524e-02  1.3515387e-02  1.5257311e-03
  1.2701781e-02 -6.8107317e-03 -1.8928028e-03  1.1537147e-02
 -1.5043275e-02 -7.8722071e-03 -1.5023164e-02 -1.8600845e-03
  1.9076237e-02 -1.4638334e-02 -4.6675373e-03 -3.8754821e-03
  1.6154874e-02 -1.1861792e-02  9.0324880e-05 -9.5074680e-03
 -1.9207101e-02  1.0014586e-02 -1.7519170e-02 -8.7836506e-03
 -7.0199967e-05 -5.9236289e-04 -1.5322480e-02  1.9229487e-02
  9.9641159e-03  1.8466286e-02]
woman: [-0.01631583  0.0089916  -0.00827415  0.00164907  0.01699724 -0.00892435
  0.009035   -0.01357392 -0.00709698  0.01879702 -0.00315531  0.00064274
 -0.00828126 -0.01536538 -0.00301602  0.00493

CBOW modeli, bir kelimenin etrafındaki bağlam (context) kelimelerine bakarak hedef kelimeyi tahmin etmeye çalışır. Daha hızlı eğitilir ve sık geçen kelimelerde iyi sonuç verir.


Skip-Gram modeli ise hedef kelimeye bakarak onun bağlamındaki kelimeleri tahmin etmeye çalışır. Özellikle seyrek görülen kelimeler için daha etkili temsil öğrenir, ancak eğitim süresi biraz daha uzundur.

human, mortal, wise: Bu kelimeler, "king" kavramına ait nitelikleri (insani özellikleri) yansıtıyor. Küçük korpusta "man", "king", "human", "mortal" gibi kelimeler yan yana geçtiği için, model bu kelimeleri vektör uzayında yakınlaştırmış.


with, is, the, a, and gibi kelimeler ise yüksek frekanslı bağlaçlar ve yardımcı fiiller olduğundan birçok cümlede geçtiği için "king" ile korelasyon göstermiş olabilir. Bu, küçük veri setlerinde yaygındır.


queen'in listede çıkması modelin semantik benzerliği algılamaya başladığını gösterir. Ancak "queen"'in skorunun düşük olması, "king" ve "queen" kelimelerinin korpusta birlikte yeterince güçlü bir bağlamda yer almadığını gösteriyor.

Sonuç:
Model "king" kelimesinin insani ve yönetici özelliklerini içeren kelimeleri doğru şekilde ilişkilendirmiş, ancak bağlamsal zenginlik sınırlı olduğu için "queen" gibi güçlü semantik eşdeğerler düşük skorda kalmış.